# STAC data prep — GUM (GeoParquet)

Converts the GUM shapefile to a single **GeoParquet** file and writes the enriched metadata next to it.

- **Case:** vector data, one layer, 911 551 polygons with 10 attribute columns, no time dimension (~1.8 GB .shp + 1.6 GB .dbf as shapefile). Source CRS: WGS84 (EPSG:4326).
- **Format:** one GeoParquet (see `DATA_FORMATS.md`, vector row; same approach as `18_hydrolakes` and `19_country_boundaries`). Rows are sorted along a Hilbert curve and a bbox covering column is written, so readers can filter spatially without loading everything.
- **OGC service:** not created. The GeoServer of the Global Coastal Atlas has no GeoParquet data store, and the shapefile is not published instead.
- **Metadata template:** `metadata/metadata_GUM.json`. It is incomplete (empty bbox, time, keywords, media type, collection id), so the missing fields are filled below.

Source: `P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\GUM_v1.0.shp`

**Running in docker:** the `P:` drive is not visible in the container, so run `docker\run_notebook_with_p_drive.ps1 -Notebook 24_gum.ipynb -Variable GUM_v1.0` from the host. It copies the shapefile parts to a local mirror, executes this notebook, copies `stac\` back to `P:`, verifies the hashes and removes the mirror. The notebook needs about 6 GB of memory.

### Import packages

In [1]:
import json
import os
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

### Define drive paths

`DATA_VIVIAN` and `STAC_DATA_REPO` are set in the docker container; otherwise the Windows host paths are used.

In [2]:
VARIABLE = "GUM_v1.0"
COLLECTION_ID = "gum"
PARQUET_NAME = "gum"

# DATA_VIVIAN is set in docker, where the P: drive is not mounted (a local mirror is used and stac/ is copied back to P:)
data_vivian = Path(os.environ.get("DATA_VIVIAN", r"P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian"))
stac_repo = Path(os.environ.get("STAC_DATA_REPO", r"C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data"))

source_shp = data_vivian / f"{VARIABLE}.shp"
metadata_template_path = stac_repo / "metadata" / "metadata_GUM.json"

processed_data_dir = data_vivian / "stac" / COLLECTION_ID
parquet_dir = processed_data_dir / "parquet"
parquet_dir.mkdir(parents=True, exist_ok=True)

for p in (source_shp, metadata_template_path):
    if not p.is_file():
        raise FileNotFoundError(p)
print("Source:", source_shp)
print("Metadata template:", metadata_template_path)
print("Output dir:", processed_data_dir)

Source: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\GUM_v1.0.shp
Metadata template: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\metadata\metadata_GUM.json
Output dir: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\gum


### Read raw data

Takes a few minutes and about 6 GB of memory.

In [3]:
gdf = gpd.read_file(source_shp, engine="pyogrio", encoding="utf-8")

empty = gdf.geometry.isna() | gdf.geometry.is_empty
print("rows:", len(gdf), "| crs:", gdf.crs.to_string() if gdf.crs else None)
print("geometry types:", gdf.geom_type.value_counts().to_dict())
print("OBJECTID unique:", gdf["OBJECTID"].is_unique, "| null or empty geometries:", int(empty.sum()))
print("total bounds:", gdf.total_bounds.tolist())
gdf.drop(columns="geometry").head()

rows: 911551 | crs: EPSG:4326
geometry types: {'Polygon': 876342, 'MultiPolygon': 35209}
OBJECTID unique: True | null or empty geometries: 0
total bounds: [-179.9999807496447, -89.98525679540779, 179.99998550428688, 83.58929179966191]


,OBJECTID,Symbol,Descriptio,XX,YY,ZZ,AA,DD,Shape_Leng,Shape_Area
0,1,Q2,Sedimentos cuaternarios,Us,nn,nn,qu,nn,8.871187,1.467516
1,2,Q2,Sedimentos cuaternarios,Us,nn,nn,qu,nn,44.650256,8.659353
2,3,Q1,Sedimentos cuaternarios,Us,nn,nn,qu,nn,18.380242,8.761808
3,4,Q,"Depositos aluviales, fluvio-lacustres, fluviog...",Us,nn,nn,qu,nn,0.501272,0.002529
4,5,Q,"Depositos aluviales, fluvio-lacustres, fluviog...",Us,nn,nn,qu,nn,0.137337,0.001071


### Check the CRS

The shapefile is in WGS84, the CRS the STAC catalog uses. A missing `.prj` is set to EPSG:4326; any other CRS is reprojected.

In [4]:
print("CRS before:", gdf.crs.to_string() if gdf.crs else None)
if gdf.crs is None:
    gdf = gdf.set_crs(4326)
elif gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(4326)
print("CRS:", gdf.crs.to_string())
print("total bounds (EPSG:4326):", gdf.total_bounds.tolist())

CRS before: EPSG:4326
CRS: EPSG:4326
total bounds (EPSG:4326): [-179.9999807496447, -89.98525679540779, 179.99998550428688, 83.58929179966191]


### Make the metadata complete (dataset dependent)

Fill the fields that are empty in the template. Column descriptions follow the GUM paper (Börker et al., 2018); please double-check the descriptions that are marked with `?` before publishing.

In [5]:
with open(metadata_template_path, encoding="utf-8") as f:
    metadata = json.load(f)

# Descriptions per attribute column (used as table:columns in the STAC item)
table_columns = {
    "OBJECTID": "Feature identifier",
    "Symbol": "Map symbol of the unit in the source map",
    "Descriptio": "Description of the unit in the source map (original language)",
    "XX": "Sediment type code (?)",
    "YY": "Sediment subtype code (?)",
    "ZZ": "Grain size / texture code (?)",
    "AA": "Age code (?)",
    "DD": "Additional attribute code, e.g. thickness or deposit characteristic (?)",
    "Shape_Leng": "Polygon perimeter (degrees, computed in EPSG:4326)",
    "Shape_Area": "Polygon area (square degrees, computed in EPSG:4326)",
    "geometry": "Unconsolidated sediment unit polygon (EPSG:4326)",
}
missing = set(gdf.columns) - set(table_columns)
assert not missing, f"Columns without description: {missing}"

west, south, east, north = (float(v) for v in gdf.total_bounds)
bbox = [max(west, -180.0), max(south, -90.0), min(east, 180.0), min(north, 90.0)]

# Extra info that the raw template does not contain
metadata.update({
    "TITLE_ABBREVIATION": "GUM",
    "HISTORY": [
        "GUM v1.0 (Börker et al., 2018), shapefile GUM_v1.0.shp, downloaded from PANGAEA",
        "Converted to a single GeoParquet (zstd, EPSG:4326, rows sorted along a Hilbert curve, bbox covering column)",
        "Geometries and attributes are unchanged"
],
    "MEDIA_TYPE": "application/vnd.apache.parquet",
    "DATA_MODEL": "vector_geoparquet",
    "DIMENSIONS": [],
    "SPATIAL_EXTENT": [round(v, 6) for v in bbox],
    # Publication year; the dataset has no time dimension
    "TEMPORAL_EXTENT": ["2018-01-01T00:00:00Z", ""],
    "KEYWORDS": ["unconsolidated sediments", "quaternary", "loess", "geology", "global"],
    "TAGS": ["sediments", "geology", "polygons", "geoparquet"],
    "LONG_NAME": "Global unconsolidated sediment polygons",
    "UNITS": "",
    "COMMENT": (
        "Vector layer with one polygon per unconsolidated sediment unit, with sediment type, subtype, grain size, age "
        "and thickness where available. The classification codes XX, YY, ZZ, AA and DD follow the GUM documentation; "
        "the descriptions marked with ? are to be confirmed. Descriptio is in the language of the source map. The "
        "temporal extent is the publication year; the layer is static."
    ),
    "AUTHOR": "Börker J, Hartmann J, Amann T, Romero-Mujalli G",
    "CRS": "EPSG:4326",
    "ITEM_BBOX_CRS": "EPSG:4326",
    "SPATIAL_RESOLUTION": None,
    "NODATA": None,
    "DATA_TYPE": "GeoParquet",
    "COLLECTION_ID": COLLECTION_ID,
    "WMS_DATASET": "",  # no OGC service: GeoServer has no GeoParquet data store
    "FEATURE_COUNT": int(len(gdf)),
    "TABLE_COLUMNS": table_columns,
})
metadata["DOI"] = "https://doi.org/10.1002/2017GC007273"  # the template holds the PANGAEA data DOI; sci:doi needs a doi.org DOI (the paper)

print("SPATIAL_EXTENT:", metadata["SPATIAL_EXTENT"])
print("FEATURE_COUNT:", metadata["FEATURE_COUNT"])

SPATIAL_EXTENT: [-179.999981, -89.985257, 179.999986, 83.589292]
FEATURE_COUNT: 911551


### Write data to GeoParquet

`zstd` compression, row groups of 100 000 features, rows ordered along a Hilbert curve so that nearby features share a row group, and a `bbox` covering column for spatial filtering. Null or empty geometries (if any) are put at the end, because they have no position on the curve.

In [6]:
valid = ~(gdf.geometry.isna() | gdf.geometry.is_empty).to_numpy()
hilbert = gdf.geometry[valid].hilbert_distance().to_numpy()
order = np.concatenate([np.flatnonzero(valid)[np.argsort(hilbert)], np.flatnonzero(~valid)])
gdf = gdf.iloc[order].reset_index(drop=True)

out_path = parquet_dir / f"{PARQUET_NAME}.parquet"
gdf.to_parquet(
    out_path,
    index=False,
    compression="zstd",
    row_group_size=100_000,
    write_covering_bbox=True,
)

metadata_output_path = processed_data_dir / f"metadata_{COLLECTION_ID}.json"
with open(metadata_output_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)

before_mb = sum(p.stat().st_size for p in source_shp.parent.glob(f"{VARIABLE}.*")) / 1e6
print(f"Wrote GeoParquet: {out_path} ({out_path.stat().st_size / 1e6:.0f} MB, shapefile parts {before_mb:.0f} MB)")
print("Wrote metadata:", metadata_output_path)

Wrote GeoParquet: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\gum\parquet\gum.parquet (1447 MB, shapefile parts 3572 MB)
Wrote metadata: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\gum\metadata_gum.json


### Verify the output

Same rows (in the same order), columns and values as the processed GeoDataFrame, EPSG:4326, readable as GeoParquet.

In [7]:
meta = pq.ParquetFile(out_path).metadata
print("row groups:", meta.num_row_groups, "| rows:", meta.num_rows)
assert meta.num_rows == len(gdf)

new = gpd.read_parquet(out_path)
assert new.crs.to_epsg() == 4326
assert list(new.columns.drop("bbox", errors="ignore")) == list(gdf.columns)

# Rows were written in the order of gdf, so they can be compared position by position
attrs = [c for c in gdf.columns if c != "geometry"]
pd.testing.assert_frame_equal(pd.DataFrame(new[attrs]), pd.DataFrame(gdf[attrs]))
same_geom = new.geometry.geom_equals_exact(gdf.geometry, tolerance=0) | (new.geometry.isna() & gdf.geometry.isna())
assert same_geom.all(), "Geometries changed"
print("OK: gum GeoParquet and metadata are ready for the STAC builder notebook.")

row groups: 10 | rows: 911551
OK: gum GeoParquet and metadata are ready for the STAC builder notebook.
